In [16]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, col, lit
spark = SparkSession.builder.appName("Jupyter").getOrCreate()

spark

events = spark.read.option("header", "true").csv("/home/iceberg/data/events.csv").withColumn("event_date", expr("DATE_TRUNC('day', event_time)"))
devices = spark.read.option("header","true").csv("/home/iceberg/data/devices.csv")

df = events.join(devices,on="device_id",how="left")
df = df.withColumnsRenamed({'browser_type': 'browser_family', 'os_type': 'os_family'})

df.join(df, lit(1) == lit(1)).take(5)

25/07/15 03:44:18 WARN Column: Constructing trivially true equals predicate, '1 = 1'. Perhaps you need to use aliases.


[Row(device_id='532630305', user_id='1037710827', referrer=None, host='www.zachwilson.tech', url='/', event_time='2021-03-08 17:27:24.241000', event_date=datetime.datetime(2021, 3, 8, 0, 0), browser_family='Other', os_family='Other', device_type='Other', device_id='532630305', user_id='1037710827', referrer=None, host='www.zachwilson.tech', url='/', event_time='2021-03-08 17:27:24.241000', event_date=datetime.datetime(2021, 3, 8, 0, 0), browser_family='Other', os_family='Other', device_type='Other'),
 Row(device_id='532630305', user_id='1037710827', referrer=None, host='www.zachwilson.tech', url='/', event_time='2021-03-08 17:27:24.241000', event_date=datetime.datetime(2021, 3, 8, 0, 0), browser_family='Other', os_family='Other', device_type='Other', device_id='532630305', user_id='925588856', referrer=None, host='www.eczachly.com', url='/', event_time='2021-05-10 11:26:21.247000', event_date=datetime.datetime(2021, 5, 10, 0, 0), browser_family='Other', os_family='Other', device_type='

In [17]:
sorted = df.repartition(10, col("event_date"))\
    .sortWithinPartitions(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sortedTwo = df.repartition(10, col("event_date"))\
    .sort(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sorted.explain()
sortedTwo.explain()


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [device_id#606, user_id#605, referrer#607, host#608, url#609, cast(event_time#610 as timestamp) AS event_time#745, event_date#617, browser_family#662, os_family#663, device_type#646]
   +- Sort [event_date#617 ASC NULLS FIRST, host#608 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(event_date#617, 10), REPARTITION_BY_NUM, [plan_id=1338]
         +- Project [device_id#606, user_id#605, referrer#607, host#608, url#609, event_time#610, event_date#617, browser_type#644 AS browser_family#662, os_type#645 AS os_family#663, device_type#646]
            +- BroadcastHashJoin [device_id#606], [device_id#643], LeftOuter, BuildRight, false
               :- Project [user_id#605, device_id#606, referrer#607, host#608, url#609, event_time#610, date_trunc(day, cast(event_time#610 as timestamp), Some(Etc/UTC)) AS event_date#617]
               :  +- FileScan csv [user_id#605,device_id#606,referrer#607,host#608,url#609,ev

In [ ]:
# .sortWithinPartitions() sorts within partitions, whereas .sort() is a global sort, which is very slow

# Note - exchange is synonymous with Shuffle

In [6]:
sorted = df.repartition(10, col("event_date"))\
    .sortWithinPartitions(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sortedTwo = df.repartition(10, col("event_date"))\
    .sort(col("event_date"), col("host"))\
    .withColumn("event_time", col("event_time").cast("timestamp")) 

sorted.explain()
sortedTwo.explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [device_id#332, user_id#331, referrer#333, host#334, url#335, cast(event_time#336 as timestamp) AS event_time#566, event_date#343, browser_family#388, os_family#389, device_type#372]
   +- Sort [event_date#343 ASC NULLS FIRST, host#334 ASC NULLS FIRST], false, 0
      +- Exchange hashpartitioning(event_date#343, 10), REPARTITION_BY_NUM, [plan_id=946]
         +- Project [device_id#332, user_id#331, referrer#333, host#334, url#335, event_time#336, event_date#343, browser_type#370 AS browser_family#388, os_type#371 AS os_family#389, device_type#372]
            +- BroadcastHashJoin [device_id#332], [device_id#369], LeftOuter, BuildRight, false
               :- Project [user_id#331, device_id#332, referrer#333, host#334, url#335, event_time#336, date_trunc(day, cast(event_time#336 as timestamp), Some(Etc/UTC)) AS event_date#343]
               :  +- FileScan csv [user_id#331,device_id#332,referrer#333,host#334,url#335,eve

In [28]:
start_df = df.repartition(4, col("event_date")).withColumn("event_time", col("event_time").cast("timestamp"))

first_sort_df = start_df.sortWithinPartitions(col("event_date"), col("browser_family"), col("host"))

start_df.write.mode("overwrite").saveAsTable("bootcamp.events_unsorted")
first_sort_df.write.mode("overwrite").saveAsTable("bootcamp.events_sorted")

In [30]:
%%sql

CREATE DATABASE IF NOT EXISTS bootcamp

++
||
++
++

In [8]:
%%sql

DROP TABLE IF EXISTS bootcamp.events

++
||
++
++

In [9]:
%%sql

DROP TABLE IF EXISTS bootcamp.events_sorted

++
||
++
++

In [35]:
%%sql

CREATE TABLE IF NOT EXISTS bootcamp.events (
    url STRING,
    referrer STRING,
    browser_family STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (years(event_date));


++
||
++
++

In [36]:
%%sql


CREATE TABLE IF NOT EXISTS bootcamp.events_sorted (
    url STRING,
    referrer STRING,
    browser_family STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (years(event_date));

++
||
++
++

In [37]:
%%sql


CREATE TABLE IF NOT EXISTS bootcamp.events_unsorted (
    url STRING,
    referrer STRING,
    browser_family STRING,
    os_family STRING,
    device_family STRING,
    host STRING,
    event_time TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (year(event_date));

++
||
++
++

In [ ]:

start_df = df.repartition(4, col("event_date")).withColumn("event_time", col("event_time").cast("timestamp")) \
    
first_sort_df = start_df.sortWithinPartitions(col("event_date"), col("host"))

start_df.write.mode("overwrite").saveAsTable("bootcamp.events_unsorted")
first_sort_df.write.mode("overwrite").saveAsTable("bootcamp.events_sorted")

In [38]:
%%sql

SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'sorted' 
FROM demo.bootcamp.events_sorted.files

UNION ALL
SELECT SUM(file_size_in_bytes) as size, COUNT(1) as num_files, 'unsorted' 
FROM demo.bootcamp.events_unsorted.files





size,num_files,sorted
5095110,4,sorted
5556664,4,unsorted


In [ ]:
%%sql 
SELECT COUNT(1) FROM bootcamp.matches_bucketed.files

In [42]:
%%sql

SELECT *
FROM demo.bootcamp.events_sorted.files

content,file_path,file_format,spec_id,partition,record_count,file_size_in_bytes,column_sizes,value_counts,null_value_counts,nan_value_counts,lower_bounds,upper_bounds,key_metadata,split_offsets,equality_ids,sort_order_id,referenced_data_file,content_offset,content_size_in_bytes,readable_metrics
0,s3://warehouse/bootcamp/events_sorted/data/00000-57-d68abd0a-790d-4c57-a670-ef1c7fb1142e-0-00001.parquet,PARQUET,1,Row(event_date_year=None),89391,1032261,"{1: 107466, 2: 61022, 3: 11455, 4: 12926, 6: 7383, 7: 426449, 8: 2293, 9: 77425, 10: 310063, 11: 10711}","{1: 89391, 2: 89391, 3: 89391, 4: 89391, 6: 89391, 7: 89391, 8: 89391, 9: 89391, 10: 89391, 11: 89391}","{1: 0, 2: 46359, 3: 0, 4: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 1, 11: 0}",{},"{1: bytearray(b'/'), 2: bytearray(b'52.20.78.240'), 3: bytearray(b'%E3%82%A6%E3%82%'), 4: bytearray(b'Android'), 6: bytearray(b'aashish.techcrea'), 7: bytearray(b' \xba\xe7\xb8\xa8\xb8\x05\x00'), 8: bytearray(b'\x00\xa0&\xb4\xa8\xb8\x05\x00'), 9: bytearray(b'-100210680'), 10: bytearray(b'-1000095488'), 11: bytearray(b'17MB150WB')}","{1: bytearray(b'/zzageqnf.php?Fp'), 2: bytearray(b'zachwilson.tech'), 3: bytearray(b'webprosbot'), 4: bytearray(b'iOS'), 6: bytearray(b'zachwilson.techd'), 7: bytearray(b'\xe8\xb0\x1b\x8ec\x03\x06\x00'), 8: bytearray(b'\x00\xe0dqO\x03\x06\x00'), 9: bytearray(b'999535123'), 10: bytearray(b'999884938'), 11: bytearray(b'vivo $2')}",None,[4],None,0,None,None,None,"Row(browser_family=Row(column_size=11455, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='%E3%82%A6%E3%82%', upper_bound='webprosbot'), device_id=Row(column_size=77425, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='-100210680', upper_bound='999535123'), device_type=Row(column_size=10711, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='17MB150WB', upper_bound='vivo $2'), event_date=Row(column_size=2293, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound=datetime.datetime(2021, 1, 12, 0, 0), upper_bound=datetime.datetime(2023, 8, 20, 0, 0)), event_time=Row(column_size=426449, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound=datetime.datetime(2021, 1, 12, 0, 1, 19, 764000), upper_bound=datetime.datetime(2023, 8, 20, 23, 59, 41, 89000)), host=Row(column_size=7383, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='aashish.techcrea', upper_bound='zachwilson.techd'), os_family=Row(column_size=12926, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='Android', upper_bound='iOS'), referrer=Row(column_size=61022, value_count=89391, null_value_count=46359, nan_value_count=None, lower_bound='52.20.78.240', upper_bound='zachwilson.tech'), url=Row(column_size=107466, value_count=89391, null_value_count=0, nan_value_count=None, lower_bound='/', upper_bound='/zzageqnf.php?Fp'), user_id=Row(column_size=310063, value_count=89391, null_value_count=1, nan_value_count=None, lower_bound='-1000095488', upper_bound='999884938'))"
0,s3://warehouse/bootcamp/events_sorted/data/00001-58-d68abd0a-790d-4c57-a670-ef1c7fb1142e-0-00001.parquet,PARQUET,1,Row(event_date_year=None),99232,1165546,"{1: 142178, 2: 67363, 3: 11914, 4: 16543, 6: 9119, 7: 475862, 8: 2373, 9: 86514, 10: 337013, 11: 11522}","{1: 99232, 2: 99232, 3: 99232, 4: 99232, 6: 99232, 7: 99232, 8: 99232, 9: 99232, 10: 99232, 11: 99232}","{1: 0, 2: 49299, 3: 0, 4: 0, 6: 0, 7: 0, 8: 0, 9: 0, 10: 58, 11: 0}",{},"{1: bytearray(b'""/?""""<?=print(93'), 2: bytearray(b'""https://www.goo'), 3: bytearray(b') Bot'), 4: bytearray(b'Android'), 6: bytearray(b'abhishekanand.te'), 7: bytearray(b'(\x83\xb2EX\xb8\x05\x00'), 8: bytearray(b'\x00 \xc9<X\xb8\x05\x00'), 9: bytearray(b'-100210680'), 10: bytearray(b'-1000370060'), 11: bytearray(b'13 Pro Max')}","{1: bytearray(b'/zz.php'), 2: bytearray(b'zachwilson.tech'), 3: bytearray(b'webprosbot'), 4: bytearray(b'iOS'), 6: bytearray(b'zsavi524.techcrf'), 7: bytearray(b'\x88\xb8\x07P;\x03\x06